
## Folder 04 — File 02: Silver Products


**Transformation rules:**

- Trim spaces from product names and categories.
- Standardize stock_status to uppercase.
- Convert unit_price and unit_cost to DECIMAL(12,2).
- Reject missing product IDs, missing names, invalid prices, and negative prices or costs.
- Keep one record per product_id.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE - SILVER PRODUCTS TRANSFORMATION
# Standardization, validation, deduplication and quarantine
# ============================================================


## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_products"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_products"
)

print("Project configuration completed.")


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze products

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run the Bronze ingestion notebook first."
    )

bronze_df = spark.table(BRONZE_TABLE)

bronze_count = bronze_df.count()

print(f"BRONZE PRODUCT RECORDS: {bronze_count}")

display(bronze_df.limit(10))


## 5. Clean and standardize product columns

required_columns = [
    "product_id",
    "product_name",
    "category",
    "stock_status",
    "unit_price",
    "unit_cost",
    "created_at"
]

missing_columns = [
    column
    for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise Exception(
        f"Required columns missing from Bronze: {missing_columns}"
    )

cleaned_df = (
    bronze_df

    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )

    .withColumn(
        "product_name",
        F.trim(F.col("product_name").cast("string"))
    )

    .withColumn(
        "category",
        F.initcap(F.trim(F.col("category").cast("string")))
    )

    .withColumn(
        "stock_status",
        F.upper(F.trim(F.col("stock_status").cast("string")))
    )

    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )

    .withColumn(
        "unit_cost",
        F.expr("try_cast(unit_cost AS DECIMAL(12,2))")
    )

    .withColumn(
        "created_at",
        F.to_date(F.col("created_at").cast("string"))
    )
)

print("Product standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid product records

invalid_condition = (
    F.col("product_id").isNull()
    | (F.col("product_id") == "")

    | F.col("product_name").isNull()
    | (F.col("product_name") == "")

    | F.col("unit_price").isNull()
    | F.col("unit_cost").isNull()

    | (F.col("unit_price") < 0)
    | (F.col("unit_cost") < 0)

    | F.col("created_at").isNull()
)


invalid_df = (
    cleaned_df
    .filter(invalid_condition)

    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                F.lit("Missing or empty product_id")
            ),

            F.when(
                F.col("product_name").isNull()
                | (F.col("product_name") == ""),
                F.lit("Missing or empty product_name")
            ),

            F.when(
                F.col("unit_price").isNull(),
                F.lit("Missing or invalid unit_price")
            ),

            F.when(
                F.col("unit_cost").isNull(),
                F.lit("Missing or invalid unit_cost")
            ),

            F.when(
                F.col("unit_price") < 0,
                F.lit("Negative unit_price")
            ),

            F.when(
                F.col("unit_cost") < 0,
                F.lit("Negative unit_cost")
            ),

            F.when(
                F.col("created_at").isNull(),
                F.lit("Missing or invalid created_at")
            )
        )
    )

)

valid_df = cleaned_df.filter(~invalid_condition)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(f"VALID PRODUCT RECORDS: {valid_count}")
print(f"INVALID PRODUCT RECORDS: {invalid_count}")

print("QUARANTINE REASONS:")

display(
    invalid_df
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)


## 7. Remove duplicate product records

# Use available ingestion metadata to select the
# latest record for each product_id.

preferred_order_columns = [
    "_ingestion_timestamp",
    "_ingested_at",
    "_source_file_modification_time",
    "_source_file"
]

order_columns = [
    column
    for column in preferred_order_columns
    if column in valid_df.columns
]

# Add business columns as deterministic tie-breakers.
for column in [
    "created_at",
    "product_name",
    "category",
    "unit_price",
    "unit_cost",
    "stock_status"
]:
    if column in valid_df.columns and column not in order_columns:
        order_columns.append(column)

if not order_columns:
    raise Exception(
        "No columns available for deterministic deduplication."
    )

deduplication_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        *[
            F.col(column).desc_nulls_last()
            for column in order_columns
        ]
    )
)

ranked_df = valid_df.withColumn(
    "_row_number",
    F.row_number().over(deduplication_window)
)

# Keep the latest record in Silver.

deduplicated_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# Preserve removed duplicate records in quarantine.

duplicate_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")

    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate product_id")
    )
)

deduplicated_count = deduplicated_df.count()
duplicate_count = duplicate_df.count()

print(f"DEDUPLICATED SILVER RECORDS: {deduplicated_count}")
print(f"DUPLICATE RECORDS: {duplicate_count}")


## 8. Combine invalid and duplicate records

quarantine_df = (
    invalid_df
    .unionByName(
        duplicate_df,
        allowMissingColumns=True
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

quarantine_count = quarantine_df.count()

print(f"TOTAL QUARANTINE RECORDS: {quarantine_count}")

display(
    quarantine_df
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)


## 9. Save quarantine records

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table written: {QUARANTINE_TABLE}")


## 10. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")


## 11. Validate Silver product records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_product_groups = (
    silver_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_price_records = (
    silver_df
    .filter(
        F.col("unit_price").isNull()
        | F.col("unit_cost").isNull()
        | (F.col("unit_price") < 0)
        | (F.col("unit_cost") < 0)
    )
    .count()
)

missing_product_ids = (
    silver_df
    .filter(
        F.col("product_id").isNull()
        | (F.col("product_id") == "")
    )
    .count()
)

missing_product_names = (
    silver_df
    .filter(
        F.col("product_name").isNull()
        | (F.col("product_name") == "")
    )
    .count()
)

missing_creation_dates = (
    silver_df
    .filter(F.col("created_at").isNull())
    .count()
)

print("=" * 55)
print("SILVER PRODUCT VALIDATION")
print("=" * 55)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate product groups: {duplicate_product_groups}")
print(f"Invalid price records: {invalid_price_records}")
print(f"Missing product IDs: {missing_product_ids}")
print(f"Missing product names: {missing_product_names}")
print(f"Missing creation dates: {missing_creation_dates}")

if (
    duplicate_product_groups > 0
    or invalid_price_records > 0
    or missing_product_ids > 0
    or missing_product_names > 0
    or missing_creation_dates > 0
):
    raise Exception("Silver product validation failed.")

print("Silver product validation passed.")


## 12. Bronze to Silver reconciliation

bronze_distinct_ids = (
    cleaned_df
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
    )
    .select("product_id")
    .distinct()
)

silver_distinct_ids = (
    silver_df
    .select("product_id")
    .distinct()
)

missing_from_silver = (
    bronze_distinct_ids
    .join(
        silver_distinct_ids,
        on="product_id",
        how="left_anti"
    )
)

missing_silver_count = missing_from_silver.count()

print("=" * 55)
print("BRONZE TO SILVER RECONCILIATION")
print("=" * 55)

print(f"Bronze total rows: {bronze_count}")

print(
    f"Bronze distinct product IDs: "
    f"{bronze_distinct_ids.count()}"
)

print(f"Silver total rows: {total_silver_records}")

print(
    f"Silver distinct product IDs: "
    f"{silver_distinct_ids.count()}"
)

print(
    f"Unique Bronze IDs missing from Silver: "
    f"{missing_silver_count}"
)

display(missing_from_silver.limit(20))


## 13. Check quarantine explanation for missing IDs

quarantine_ids = (
    quarantine_df
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
)

unexplained_missing_ids = (
    missing_from_silver
    .join(
        quarantine_ids,
        on="product_id",
        how="left_anti"
    )
)

unexplained_count = unexplained_missing_ids.count()

print(
    f"Missing IDs without quarantine explanation: "
    f"{unexplained_count}"
)

display(unexplained_missing_ids.limit(20))


## 14. Final summary

print("=" * 55)
print("PRODUCT SILVER TRANSFORMATION COMPLETED")
print("=" * 55)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Quarantine table: {QUARANTINE_TABLE}")

print(f"Bronze records: {bronze_count}")
print(f"Valid records before deduplication: {valid_count}")
print(f"Invalid records: {invalid_count}")
print(f"Duplicate records quarantined: {duplicate_count}")
print(f"Silver unique products: {total_silver_records}")
print(f"Total quarantine records: {quarantine_count}")
print(f"Unaccounted missing product IDs: {unexplained_count}")

if unexplained_count > 0:
    raise Exception(
        "Some missing product IDs have no quarantine explanation."
    )

print("=" * 55)
print("FINAL SILVER PRODUCT VALIDATION PASSED")
print("=" * 55)

display(
    silver_df
    .orderBy("product_id")
    .limit(20)
)
